# Geneformer: single-cell foundation model embeddings

This notebook explains:
1. What Geneformer is and how it was pre-trained
2. What **zero-shot embedding extraction** means
3. How cell-level embeddings were extracted for the Onek1K dataset
4. How we **aggregated** 1.18 M cell embeddings → 981 donor-level feature vectors
5. How to load and inspect the final pseudobulk embedding files

**No GPU is required for this notebook.** Embedding extraction (the expensive step) has already been done; we only load and inspect the pre-computed files here.

---

## 1 · What is a foundation model?

A **foundation model** is a large neural network pre-trained on a massive, general-purpose dataset. After pre-training it can be applied to many downstream tasks — sometimes without any further training at all (zero-shot), or with minimal task-specific fine-tuning.

The concept originated in NLP (GPT, BERT) and has since been applied to images, protein sequences, and — most recently — **single-cell transcriptomics**.

```
                  ┌─────────────────────────────┐
                  │   Pre-training dataset       │
                  │   (30 M cells, diverse)      │
                  └──────────────┬──────────────┘
                                 │  self-supervised
                                 ▼  learning
                  ┌─────────────────────────────┐
                  │   Foundation model weights   │
                  │   (Geneformer)               │
                  └──────────────┬──────────────┘
                    ┌────────────┼────────────┐
                    ▼            ▼            ▼
              Cell-type    Gene network   Age / disease
              annotation   inference      prediction
              (zero-shot)  (zero-shot)    (fine-tuned)
```

---

## 2 · Geneformer architecture

**Reference:** Theodoris et al., *Nature* 2023 — *"Transfer learning enables predictions in network biology"*

### Input representation

Geneformer does **not** use raw count values. Instead, each cell is represented as an **ordered list of genes, ranked by their expression level** (highest → lowest after normalisation).

```
Raw counts (cell):          Token sequence (rank-ordered):
  ACTB   = 1 200      →      [ACTB, RPS27, B2M, MALAT1, …, <low-expr genes>]
  RPS27  = 980        →      Token IDs: [2048, 9731, 4403, 872, …]
  B2M    = 810
  MALAT1 = 650
  …
```

This rank-based tokenisation makes the model robust to:
- Differences in sequencing depth between cells
- Batch effects (library size variation)
- Platform differences (10x v2 vs v3, etc.)

### Transformer encoder

Geneformer uses a **BERT-style encoder** (bidirectional transformer). During pre-training, ~15% of gene tokens are randomly masked and the model learns to predict them from context — i.e. it learns which genes tend to be co-expressed and in which biological states.

```
Token sequence:  [CLS]  ACTB  RPS27  [MASK]  MALAT1  …  [EOS]
                   │      │      │      │        │
                   ▼      ▼      ▼      ▼        ▼
              ┌────────────────────────────────────┐
              │   Transformer layers (6 × 256-dim) │
              └────────────────────────────────────┘
                   │      │      │      │        │
                   ▼      ▼      ▼      ▼        ▼
              Contextualised embeddings (256-dim each)
```

After pre-training the **[CLS] token embedding** (or the mean of all gene token embeddings, excluding [CLS] and [EOS]) represents the **cell state** as a single 256-dimensional vector.

> **In the version used for this project** (BioNeMo/NV-Geneformer), the hidden dimension was expanded to **1,152 dimensions**, giving richer representations.

---

## 3 · Zero-shot embedding extraction

"Zero-shot" means we use the **pre-trained weights without any fine-tuning** for our task.

We do not tell Geneformer what age prediction is. We simply pass each cell's gene-rank sequence through the network and extract the **internal representation** that has been learned by the self-supervised pre-training. This representation encodes the cell's transcriptional state.

```
  Cell from donor (age 72, CD8 T cell)
          │
          ▼  rank-based tokenisation
  [CLS, GZMB, GNLY, PRF1, IFNG, NKG7, …]   ← top expressed genes in this CD8 T cell
          │
          ▼  Geneformer encoder (frozen weights, no fine-tuning)
  [0.12, -0.43, 0.71, …, 0.05]              ← 1,152-dimensional embedding
          │
          └─ saved to parquet
```

### Why this works

The pre-training objective forced Geneformer to learn a compressed representation of gene co-expression across 30 million diverse cells. Biological signals like **immune activation, senescence, differentiation state, and stress response** are all encoded in the embedding — because they required learning these patterns to predict masked genes.

Age-related transcriptomic changes (inflammaging, CD8 exhaustion, clonal haematopoiesis) are captured **implicitly**, even though the model was never explicitly trained to predict age.

---

## 4 · Extraction pipeline for Onek1K

The embeddings were generated using NVIDIA BioNeMo's Geneformer implementation, submitted as a SLURM job on the Iridis HPC cluster.

**Input:** `processed_data.h5ad` — 1,183,459 cells from 981 donors (all splits combined), with `cell_id`, `donor_id`, `celltype`, `split`, `age` in `.obs`.

**What the job did:**
```
for split in [train, eval, test]:
    filter cells → split subset
    tokenise each cell (rank-order genes)
    pass through Geneformer encoder
    extract mean of gene-token embeddings (excluding CLS/EOS)
    save cell_id + 1,152-dim embedding → parquet
```

**Run time and cell counts:**

| Split | Cells | Wall time |
|-------|-------|-----------|
| train | 942,045 | ~3h 22m |
| val (eval) | 116,683 | ~27m |
| test | 124,731 | ~29m |
| **Total** | **1,183,459** | **~4h 22m** |

**Output files:**
```
cytokines-BioCreator/embeddings_onek1k_zeroshot/
  geneformer_zeroshot_onek1k_train.parquet   # 942 k rows × 1,155 cols
  geneformer_zeroshot_onek1k_val.parquet     # 117 k rows × 1,155 cols
  geneformer_zeroshot_onek1k_test.parquet    # 125 k rows × 1,155 cols
```

Each row = one cell. Columns: `0`–`1151` (embedding dims) + `cell_id`, `split`, `age`.

---

## 5 · From cell-level to donor-level: the aggregation problem

Geneformer produces **one 1,152-dim vector per cell**. But our prediction target is **one age value per donor**. We have ~1,200 cells per donor across 5 cell types.

```
Donor 42 (age 65):
  B cells:    [cell_1, cell_2, …, cell_180]    → 180 × 1,152 embeddings
  CD4 T:      [cell_181, …, cell_420]          → 240 × 1,152 embeddings
  CD8 T:      [cell_421, …, cell_630]          → 210 × 1,152 embeddings
  NK:         [cell_631, …, cell_780]          → 150 × 1,152 embeddings
  monocytes:  [cell_781, …, cell_940]          → 160 × 1,152 embeddings
  ─────────────────────────────────────────────────────
  Total: ~940 cells → need to produce ONE feature vector
```

### Strategy: mean-pool per cell type, then concatenate

**Why not pool all cells together?**  
Each cell type has a distinct transcriptional programme. A CD8 T cell embedding encodes cytotoxic state; a monocyte embedding encodes inflammatory state. Mixing them would destroy this cell-type-specific information.

**The approach:**
1. For each donor × cell type combination: compute the **mean** across all cells → one 1,152-dim vector
2. Concatenate the 5 cell-type mean vectors → one **5,760-dim** vector per donor

```
Donor 42:
  mean(B cells embeddings)    = [0.12, -0.43, …]   1,152 dims
  mean(CD4 T embeddings)      = [0.33,  0.21, …]   1,152 dims
  mean(CD8 T embeddings)      = [-0.51, 0.88, …]   1,152 dims  ← most informative!
  mean(NK embeddings)         = [0.07, -0.12, …]   1,152 dims
  mean(monocytes embeddings)  = [0.44,  0.09, …]   1,152 dims
  ─────────────────────────────────────────────────────
  concat → [0.12, -0.43, …, 0.33, 0.21, …, …, 0.44, 0.09, …]   5,760 dims
```

Column naming in the output file: `{cell_type}__emb{dim}`, e.g. `CD8_T__emb0`, `B_cells__emb512`.

---

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 110

# Project root — works whether you run from notebooks/ or the repo root
NB_DIR = Path().resolve()
if (NB_DIR / 'models' / 'train_age_model.py').exists():
    PROJ_ROOT = NB_DIR
elif (NB_DIR.parent / 'models' / 'train_age_model.py').exists():
    PROJ_ROOT = NB_DIR.parent
else:
    PROJ_ROOT = NB_DIR.parent if NB_DIR.name == 'notebooks' else NB_DIR

DATA_DIR = PROJ_ROOT / 'data'
RESULTS_DIR = PROJ_ROOT / 'results'
(RESULTS_DIR / 'figures').mkdir(parents=True, exist_ok=True)

GF_DIR = DATA_DIR / 'geneformer'
# Cell-level parquets (large). On Iridis, copy or bind shared data — includes train/val/test under data/geneformer_parquet/
PARQUET_DIR = DATA_DIR / 'geneformer_parquet'

print('Project root:', PROJ_ROOT)
print('Data directory:', DATA_DIR)
print('Aggregated embeddings (TSV):', GF_DIR)
print('Results directory:', RESULTS_DIR)
print('Cell-level parquet dir:', PARQUET_DIR)

## 6 · Inspect the raw cell-level parquet

The parquet files store one row per cell with 1,152 embedding dimensions plus metadata columns.

**Competition bundle:** `data/geneformer_parquet/` contains `train`, `val`, and `test` splits. The **test** file has **no `age` column** (labels are withheld). Train and val still include `age` for exploration.

In [ ]:
from IPython.display import display
import pyarrow.parquet as pq

val_pq = PARQUET_DIR / 'geneformer_zeroshot_onek1k_val.parquet'

if val_pq.exists():
    schema = pq.read_schema(str(val_pq))
    meta_cols = ['cell_id', 'split', 'age']
    emb_cols  = [str(i) for i in range(1152)]
    print(f'Total columns : {len(schema.names)}')
    print(f'Embedding dims: {len(emb_cols)}')
    print(f'Metadata cols : {meta_cols}')
    print(f'\nFirst 5 column names: {schema.names[:5]}')
    print(f'Last  5 column names: {schema.names[-5:]}')
else:
    print('Parquet file not accessible from this environment.')
    print('Schema: 1,155 columns = emb_0 … emb_1151 + cell_id + split + age')

In [ ]:
# Read a small sample of the val parquet (metadata columns only — fast)
if val_pq.exists():
    sample = pq.read_table(str(val_pq), columns=['cell_id', 'split', 'age']).to_pandas()
    print(f'Rows (cells): {len(sample):,}')
    print(f'Split values: {sample["split"].unique()}')
    print(f'Age range: {sample["age"].astype(float).min():.0f} – {sample["age"].astype(float).max():.0f} years')
    display(sample.head(5))
else:
    print('Example output (val split, first 5 rows):')
    print('  cell_id       split   age')
    print('  cell_948775   eval    72.0')
    print('  cell_954286   eval    49.0')
    print('  cell_986638   eval    87.0')

## 7 · The aggregation script (`aggregate_geneformer_embeddings.py`)

The script `data_prep/aggregate_geneformer_embeddings.py` implements the mean-pooling strategy.
Here is a simplified version of its core logic:

In [ ]:
# Pseudocode illustration of the aggregation (does not run on full data)
aggregation_pseudocode = """
STEP 1 — Load cell metadata from processed_data.h5ad
         columns used: cell_id, donor_id, celltype, split, age

STEP 2 — For each split (train / val / test):
         a) Load parquet: emb_0 … emb_1151 + cell_id   (skip age/split from parquet)
         b) JOIN with metadata on cell_id
            → each row now has: donor_id, celltype, emb_0 … emb_1151

STEP 3 — GROUP BY (donor_id, celltype)
         MEAN over all embedding dimensions
            → shape: (n_donor_celltype_pairs, 1,152)
            Example val: 95 donors × 5 types = 475 rows

STEP 4 — PIVOT: one row per donor, columns = {celltype}__emb{dim}
            → shape: (n_donors, 5 × 1,152) = (n_donors, 5,760)

STEP 5 — Attach split column; attach age (EXCLUDED for test donors)

STEP 6 — Save as gzip-compressed TSV
"""
print(aggregation_pseudocode)

In [ ]:
# ── Toy demonstration of the mean-pooling aggregation ──────────────────────
from IPython.display import display
np.random.seed(42)

CELL_TYPES = ['B_cells', 'CD4_T', 'CD8_T', 'NK', 'monocytes']
EMB_DIM    = 8   # using 8 dims instead of 1,152 for readability

# Simulate cell-level data for 3 donors, ~6 cells each across 3 cell types
records = []
for donor in [1, 2, 3]:
    for ct in CELL_TYPES[:3]:   # 3 cell types for brevity
        for _ in range(np.random.randint(3, 8)):   # random number of cells
            emb = np.random.randn(EMB_DIM).round(3)
            row = {'donor_id': donor, 'celltype': ct}
            row.update({f'emb{i}': emb[i] for i in range(EMB_DIM)})
            records.append(row)

cells = pd.DataFrame(records)
emb_cols_toy = [f'emb{i}' for i in range(EMB_DIM)]

print(f'Cell-level DataFrame: {cells.shape[0]} cells × {cells.shape[1]} columns')
display(cells.head(4))

In [ ]:
# STEP 3: group by (donor_id, celltype), mean-pool embeddings
group_mean = (
    cells.groupby(['donor_id', 'celltype'])[emb_cols_toy]
    .mean()
    .reset_index()
)
print(f'After mean-pooling: {group_mean.shape[0]} rows (donor × cell type pairs)')
display(group_mean.head(6).round(3))

In [ ]:
# STEP 4: pivot to wide format — one row per donor
ct_frames = []
for ct in cells['celltype'].unique():
    ct_rows = group_mean[group_mean['celltype'] == ct].copy()
    ct_rows = ct_rows.rename(columns={c: f'{ct}__{c}' for c in emb_cols_toy})
    ct_rows = ct_rows.drop(columns=['celltype']).set_index('donor_id')
    ct_frames.append(ct_rows)

wide = pd.concat(ct_frames, axis=1).reset_index()
n_feat = len(wide.columns) - 1   # exclude donor_id
print(f'Wide donor-level DataFrame: {wide.shape[0]} donors × {n_feat} features')
print(f'(In the real data: 981 donors × 5,760 features)')
display(wide.round(3))

## 8 · Load the real aggregated embeddings

In [ ]:
from IPython.display import display

val_tsv = GF_DIR / 'geneformer_pseudobulk_val.tsv.gz'

if val_tsv.exists():
    gf_val = pd.read_csv(str(val_tsv), sep='\t')
    emb_feat_cols = [c for c in gf_val.columns if '__emb' in c]

    print('── Val split ─────────────────────────────────────────')
    print(f'  Shape          : {gf_val.shape[0]} donors × {gf_val.shape[1]} columns')
    print(f'  Feature cols   : {len(emb_feat_cols):,} ({len(emb_feat_cols)} embedding dimensions)')
    print(f'  Metadata cols  : donor_id, split, age')
    print(f'  Age range      : {gf_val["age"].min():.0f} – {gf_val["age"].max():.0f} years')
    print(f'  First 4 feature columns : {emb_feat_cols[:4]}')
    print(f'  Last  4 feature columns : {emb_feat_cols[-4:]}')
    display(gf_val[['donor_id', 'split', 'age'] + emb_feat_cols[:3]].head(5))
else:
    print('Aggregated file not found at', val_tsv)
    print('Run: python data_prep/aggregate_geneformer_embeddings.py --splits val')

In [ ]:
# Summary table of all three splits
from IPython.display import display

rows = []
for split in ['train', 'val', 'test']:
    f = GF_DIR / f'geneformer_pseudobulk_{split}.tsv.gz'
    if f.exists():
        df = pd.read_csv(str(f), sep='\t', nrows=1)
        n_donors = sum(1 for _ in open(str(f), 'rb')) - 1   # rough row count via lines
        # faster: read only donor_id col
        n_donors = pd.read_csv(str(f), sep='\t', usecols=['donor_id']).shape[0]
        n_feat = len([c for c in df.columns if '__emb' in c])
        has_age = 'age' in df.columns
        rows.append({'split': split, 'donors': n_donors,
                     'features': n_feat, 'age_column': has_age})
    else:
        rows.append({'split': split, 'donors': '?', 'features': 5760,
                     'age_column': split != 'test'})

summary = pd.DataFrame(rows)
print('Aggregated Geneformer pseudobulk files:')
display(summary)

## 9 · Visualise embedding structure

Let's look at the distribution of mean embedding values across cell types, and check how age correlates with a few dimensions.

In [ ]:
from IPython.display import display

if val_tsv.exists():
    cell_types_ordered = ['B_cells', 'CD4_T', 'CD8_T', 'NK', 'monocytes']
    mean_variances = {}
    for ct in cell_types_ordered:
        cols = [c for c in gf_val.columns if c.startswith(ct + '__emb')]
        mean_variances[ct] = gf_val[cols].var(axis=0).mean()

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Left: mean variance per cell type
    ax = axes[0]
    ax.bar(cell_types_ordered,
           [mean_variances[ct] for ct in cell_types_ordered],
           color=['#4C72B0','#DD8452','#55A868','#C44E52','#8172B3'])
    ax.set_title('Mean embedding variance across donors\n(per cell type)', fontsize=11)
    ax.set_ylabel('Mean variance')
    ax.set_xlabel('Cell type')
    ax.tick_params(axis='x', rotation=20)

    # Right: scatter — CD8_T emb0 vs age
    ax2 = axes[1]
    x = gf_val['CD8_T__emb0']
    y = gf_val['age'].astype(float)
    ax2.scatter(x, y, alpha=0.7, edgecolors='white', linewidths=0.4,
                c=y, cmap='RdYlBu_r', s=60)
    from scipy.stats import pearsonr
    r, p = pearsonr(x, y)
    ax2.set_title(f'CD8_T emb dim 0 vs age\n(Pearson r={r:.2f}, p={p:.2e})', fontsize=11)
    ax2.set_xlabel('CD8_T__emb0 value')
    ax2.set_ylabel('Age (years)')

    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'figures' / 'geneformer_embedding_overview.png',
                dpi=130, bbox_inches='tight')
    plt.show()
    print('Figure saved to results/figures/geneformer_embedding_overview.png')
else:
    print('Load aggregated val file first (cell above).')

In [ ]:
# Find the embedding dimensions most correlated with age
from IPython.display import display

if val_tsv.exists():
    from scipy.stats import pearsonr

    ages = gf_val['age'].astype(float).values
    corrs = []
    for col in emb_feat_cols:
        r, _ = pearsonr(gf_val[col].fillna(0).values, ages)
        corrs.append({'feature': col, 'pearson_r': r, 'abs_r': abs(r)})

    corr_df = pd.DataFrame(corrs).sort_values('abs_r', ascending=False)
    print('Top 15 embedding dimensions most correlated with age (val set):')
    display(corr_df.head(15)[['feature', 'pearson_r']].reset_index(drop=True))

    # Count how many top-20 come from each cell type
    top20 = corr_df.head(20)
    top20['cell_type'] = top20['feature'].str.split('__').str[0]
    print('\nTop-20 by cell type:')
    display(top20['cell_type'].value_counts().reset_index())
else:
    print('Aggregated val file required.')

## 10 · Feature importance from the Random Forest

After training a Random Forest on the Geneformer pseudobulk features, feature importances confirm the biological story:

| Rank | Feature (cell-type block) | Importance | Biology |
|------|--------------------------|------------|---------|
| 1 | **CD8 T** | **0.484** | Immunosenescence, clonal expansion, effector memory accumulation with age |
| 2 | NK cells | 0.064 | NK cell dysfunction in aging |
| 3 | Monocytes | 0.063 | Inflammaging — monocyte activation increases with age |
| 4 | CD4 T | 0.061 | CD4 exhaustion, regulatory T cell changes |
| 5 | B cells | 0.050 | Age-associated B cells (ABCs) accumulate with age |

**All 5 Geneformer cell-type blocks together account for ~72% of model importance**, while the remaining 28% comes from 10,000 pseudobulk gene expression features. This demonstrates the exceptional information density of the foundation model embeddings.

The dominance of **CD8 T cells** is consistent with decades of immunogerontology research: the shift from naïve to terminally differentiated effector memory (TEMRA) CD8 T cells is one of the most reliable hallmarks of immune aging.

In [ ]:
from IPython.display import display

# Recreate the feature importance chart from the combined model run
feat_imp_data = [
    ('CD8_T (Geneformer)',    0.484, '#C44E52'),
    ('NK (Geneformer)',       0.064, '#C44E52'),
    ('monocytes (Geneformer)',0.063, '#C44E52'),
    ('CD4_T (Geneformer)',    0.061, '#C44E52'),
    ('B_cells (Geneformer)',  0.050, '#C44E52'),
    ('ENSG00000245573 (pseudobulk)', 0.008, '#4C72B0'),
    ('ENSG00000169862 (pseudobulk)', 0.006, '#4C72B0'),
    ('ENSG00000172175 (pseudobulk)', 0.005, '#4C72B0'),
    ('other pseudobulk genes',       0.199, '#4C72B0'),
]

fig, ax = plt.subplots(figsize=(9, 5))
labels = [x[0] for x in feat_imp_data]
values = [x[1] for x in feat_imp_data]
colors = [x[2] for x in feat_imp_data]

bars = ax.barh(labels[::-1], values[::-1], color=colors[::-1], edgecolor='white', linewidth=0.5)
ax.set_xlabel('Aggregated feature importance')
ax.set_title('Top features in pseudobulk + Geneformer Random Forest\n(importance summed per gene / cell-type block)', fontsize=11)

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#C44E52', label='Geneformer cell-type block'),
                   Patch(facecolor='#4C72B0', label='Pseudobulk gene expression')]
ax.legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'figures' / 'geneformer_feature_importance.png',
            dpi=130, bbox_inches='tight')
plt.show()

## Summary

| Step | What happened | Key number |
|------|--------------|------------|
| Pre-training | Geneformer trained on 30 M diverse cells | 1,152-dim cell state vector |
| Tokenisation | Each cell → rank-ordered gene list | ~2,000 tokens per cell |
| Zero-shot extraction | No fine-tuning; forward pass only | 1.18 M cells × 1,152 dims |
| Mean-pooling | Per donor × cell type | 981 × 5 = 4,905 group means |
| Final feature matrix | Flattened | 981 donors × **5,760 features** |
| Model performance | Geneformer-only Random Forest (100 trees) | **MAE 8.80, R² 0.51** |

**Key takeaway:** A 1,152-dimensional zero-shot embedding per cell type, obtained without any task-specific training, outperforms 10,000 carefully selected pseudobulk gene expression features for age prediction. This highlights the power of foundation models in single-cell biology.

---

### Files referenced

| File | Description |
|------|-------------|
| `data_prep/aggregate_geneformer_embeddings.py` | Aggregation script |
| `data/geneformer/geneformer_pseudobulk_{train,val,test}.tsv.gz` | Donor-level feature files |
| `data/geneformer_parquet/geneformer_zeroshot_onek1k_{train,val,test}.parquet` | Cell-level embeddings (test split has no `age`) |
| `models/train_age_model.py --geneformer-only` | Train RF on Geneformer features |
| `models/train_age_model.py --geneformer` | Pseudobulk + Geneformer combined |